In [ ]:
# Experiment 5: Transfer Learning using Pre-trained CNN (ResNet50)
# Improved Version — Higher Accuracy

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# ── 1. Load & Preprocess Dataset ─────────────────────────────
print("Loading CIFAR-10...")
(x_train, y_train), (x_test, y_test) = cifar10.load_data()

# Resize to 128x128 (good balance of speed vs accuracy)
x_train = tf.image.resize(x_train, (128, 128)).numpy()
x_test  = tf.image.resize(x_test,  (128, 128)).numpy()

# Use ResNet's own preprocessor (better than manual /255)
x_train = preprocess_input(x_train)
x_test  = preprocess_input(x_test)

y_train = to_categorical(y_train, 10)
y_test  = to_categorical(y_test,  10)

print(f"Train: {x_train.shape} | Test: {x_test.shape}")

# ── 2. Data Augmentation ─────────────────────────────────────
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
    tf.keras.layers.RandomContrast(0.1),
], name="augmentation")

# ── 3. Load Pre-trained ResNet50 ─────────────────────────────
print("Loading ResNet50 with ImageNet weights...")
base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(128, 128, 3))
print(f"Total base layers: {len(base_model.layers)}")

# ── 4. Freeze Base Layers ────────────────────────────────────
base_model.trainable = False

# ── 5. Build Improved Classifier Head ────────────────────────
inputs = tf.keras.Input(shape=(128, 128, 3))
x = data_augmentation(inputs)
x = base_model(x, training=False)
x = GlobalAveragePooling2D()(x)
x = BatchNormalization()(x)
x = Dense(512, activation='relu')(x)
x = Dropout(0.5)(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.3)(x)
output = Dense(10, activation='softmax')(x)

model = Model(inputs=inputs, outputs=output)

trainable_params = sum(np.prod(v.shape) for v in model.trainable_weights)
print(f"Trainable parameters: {trainable_params:,}")

# ── 6. Callbacks ─────────────────────────────────────────────
callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=4,
                  restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                      patience=2, min_lr=1e-7, verbose=1)
]

# ── 7. Phase 1 — Train Head Only ─────────────────────────────
print("\n--- Phase 1: Training Head Only (Base Frozen) ---")
model.compile(optimizer=Adam(0.001),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

history_head = model.fit(
    x_train, y_train,
    epochs=15,
    batch_size=64,
    validation_data=(x_test, y_test),
    callbacks=callbacks,
    verbose=1
)

head_epochs = len(history_head.history['accuracy'])
print(f"Phase 1 stopped at epoch {head_epochs}")

# ── 8. Phase 2 — Fine-tune Last 50 Layers ───────────────────
print("\n--- Phase 2: Fine-tuning Last 50 Layers ---")
base_model.trainable = True
for layer in base_model.layers[:-50]:
    layer.trainable = False

# Keep BatchNorm layers frozen to preserve learned statistics
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

trainable_params = sum(np.prod(v.shape) for v in model.trainable_weights)
print(f"Trainable parameters now: {trainable_params:,}")

model.compile(optimizer=Adam(1e-4),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

history_ft = model.fit(
    x_train, y_train,
    epochs=20,
    batch_size=64,
    validation_data=(x_test, y_test),
    callbacks=callbacks,
    verbose=1
)

ft_epochs = len(history_ft.history['accuracy'])
print(f"Phase 2 stopped at epoch {ft_epochs}")

# ── 9. Phase 3 — Deep Fine-tune All Layers ───────────────────
print("\n--- Phase 3: Deep Fine-tune (All Layers, Very Low LR) ---")
base_model.trainable = True

# Keep BatchNorm frozen
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

model.compile(optimizer=Adam(1e-5),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

history_deep = model.fit(
    x_train, y_train,
    epochs=15,
    batch_size=64,
    validation_data=(x_test, y_test),
    callbacks=callbacks,
    verbose=1
)

deep_epochs = len(history_deep.history['accuracy'])
print(f"Phase 3 stopped at epoch {deep_epochs}")

# ── 10. Final Evaluation ──────────────────────────────────────
loss, acc = model.evaluate(x_test, y_test, verbose=0)
print(f"\nFinal Test Accuracy : {acc*100:.2f}%")
print(f"Final Test Loss     : {loss:.4f}")

# ── 11. Plot Results ──────────────────────────────────────────
h_acc  = history_head.history['accuracy']
h_val  = history_head.history['val_accuracy']
h_loss = history_head.history['loss']
h_vloss= history_head.history['val_loss']

f_acc  = history_ft.history['accuracy']
f_val  = history_ft.history['val_accuracy']
f_loss = history_ft.history['loss']
f_vloss= history_ft.history['val_loss']

d_acc  = history_deep.history['accuracy']
d_val  = history_deep.history['val_accuracy']
d_loss = history_deep.history['loss']
d_vloss= history_deep.history['val_loss']

e1 = range(1, head_epochs + 1)
e2 = range(head_epochs + 1, head_epochs + ft_epochs + 1)
e3 = range(head_epochs + ft_epochs + 1, head_epochs + ft_epochs + deep_epochs + 1)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle('Experiment 5: Transfer Learning — ResNet50 on CIFAR-10 (Improved)', fontsize=14)

# Accuracy
axes[0].plot(e1, h_acc,  'b-o',  markersize=4, label='Phase1 Train')
axes[0].plot(e1, h_val,  'b--o', markersize=4, label='Phase1 Val')
axes[0].plot(e2, f_acc,  'r-s',  markersize=4, label='Phase2 Train')
axes[0].plot(e2, f_val,  'r--s', markersize=4, label='Phase2 Val')
axes[0].plot(e3, d_acc,  'g-^',  markersize=4, label='Phase3 Train')
axes[0].plot(e3, d_val,  'g--^', markersize=4, label='Phase3 Val')
axes[0].axvline(x=head_epochs + 0.5,              color='gray',   linestyle=':', linewidth=1.5)
axes[0].axvline(x=head_epochs + ft_epochs + 0.5,  color='purple', linestyle=':', linewidth=1.5)
axes[0].set_title('Accuracy per Phase')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

# Loss
axes[1].plot(e1, h_loss,  'b-o',  markersize=4, label='Phase1 Train')
axes[1].plot(e1, h_vloss, 'b--o', markersize=4, label='Phase1 Val')
axes[1].plot(e2, f_loss,  'r-s',  markersize=4, label='Phase2 Train')
axes[1].plot(e2, f_vloss, 'r--s', markersize=4, label='Phase2 Val')
axes[1].plot(e3, d_loss,  'g-^',  markersize=4, label='Phase3 Train')
axes[1].plot(e3, d_vloss, 'g--^', markersize=4, label='Phase3 Val')
axes[1].axvline(x=head_epochs + 0.5,              color='gray',   linestyle=':', linewidth=1.5)
axes[1].axvline(x=head_epochs + ft_epochs + 0.5,  color='purple', linestyle=':', linewidth=1.5)
axes[1].set_title('Loss per Phase')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# ── 12. Summary ───────────────────────────────────────────────
print("\n" + "="*45)
print("              RESULT SUMMARY")
print("="*45)
print(f"  Phase 1 - Head only Val Acc  : {max(h_val)*100:.2f}%")
print(f"  Phase 2 - Fine-tune Val Acc  : {max(f_val)*100:.2f}%")
print(f"  Phase 3 - Deep Fine-tune Acc : {max(d_val)*100:.2f}%")
print(f"  Final   - Test Accuracy      : {acc*100:.2f}%")
print(f"  Total Improvement            : +{(acc - max(h_val))*100:.2f}%")
print("="*45)

Loading CIFAR-10...
